# Policy Gradient Learning (REINFORCE)

This notebook walks through **policy gradient reinforcement learning**, starting from the theory and ending with a working REINFORCE agent that learns to balance the CartPole-v1 environment.

We will cover:
1. The policy gradient objective and its gradient estimator
2. REINFORCE (Monte-Carlo policy gradient) with return normalization
3. A variance-reduced variant with a learned value baseline
4. Training, evaluation, and visualization

Runs on Google Colab (CPU is fine for CartPole).

## 1. Background

A **policy** $\pi_\theta(a\mid s)$ is a distribution over actions parameterized by $\theta$. The RL objective is the expected return

$$J(\theta) = \mathbb{E}_{\tau\sim\pi_\theta}\Big[\sum_{t=0}^{T} \gamma^t r_t \Big].$$

The **policy gradient theorem** gives

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau\sim\pi_\theta}\Big[\sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t\mid s_t)\; G_t \Big],$$

where $G_t = \sum_{k=t}^{T} \gamma^{k-t} r_k$ is the discounted return from step $t$.

**REINFORCE** is the Monte-Carlo estimator of this gradient. Variance is reduced by subtracting a baseline $b(s_t)$ (often a learned value $V_\phi(s_t)$), giving the advantage $A_t = G_t - b(s_t)$.

## 2. Setup

Install dependencies. `gymnasium` is the maintained fork of OpenAI Gym.

In [ ]:
# Install project dependencies into the current runtime (quiet mode).
# gymnasium[classic-control] = the CartPole environment + rendering deps.
# torch                      = neural network + autograd.
# numpy                      = array math.
# matplotlib                 = plotting the learning curve.
!pip install -q gymnasium[classic-control] torch numpy matplotlib

In [ ]:
# --- Standard library imports ---
import random                          # Python stdlib RNG; we seed it for reproducibility.
from collections import deque          # Fixed-size FIFO, used to track a rolling mean of returns.

# --- Third-party imports ---
import gymnasium as gym                # The RL environment library (successor to OpenAI Gym).
import matplotlib.pyplot as plt        # Plotting the learning curve.
import numpy as np                     # Array math, used for observations and moving averages.
import torch                           # Tensors + autograd.
import torch.nn as nn                  # Layer/model base classes.
import torch.nn.functional as F        # Stateless ops (mse_loss, etc.).
from torch.distributions import Categorical  # Discrete probability distribution over actions.

# --- Reproducibility: fix all RNGs to the same value ---
SEED = 42                              # Arbitrary fixed integer; any value works.
random.seed(SEED)                      # Seeds Python's stdlib RNG.
np.random.seed(SEED)                   # Seeds NumPy's global RNG.
torch.manual_seed(SEED)                # Seeds PyTorch (CPU + MPS/CUDA if available).

# --- Device selection: prefer CUDA > MPS (Apple Silicon) > CPU ---
if torch.cuda.is_available():
    device = torch.device('cuda')      # NVIDIA GPU via CUDA.
elif torch.backends.mps.is_available():
    device = torch.device('mps')       # Apple Silicon GPU via Metal.
else:
    device = torch.device('cpu')       # Fallback CPU (fine for CartPole).
print('Using device:', device)         # Log which backend we're using.

## 3. Environment

CartPole-v1: 4-dim continuous state, 2 discrete actions (push left/right). The episode ends when the pole falls or after 500 steps; every step yields reward 1.

In [ ]:
# Create the CartPole environment; v1 has max 500 steps (v0 had 200).
env = gym.make('CartPole-v1')

# Size of one observation vector; used to build the policy network input layer.
obs_dim = env.observation_space.shape[0]

# Number of discrete actions; used to build the policy network output layer.
n_actions = env.action_space.n

# --- Environment metadata (from the gym registry spec) ---
print('=== Environment ===')
# Registered id of the environment (e.g. "CartPole-v1").
print(f'id                 : {env.spec.id}')
# Hard cap on steps per episode; env returns truncated=True when reached.
print(f'max_episode_steps  : {env.spec.max_episode_steps}')
# Average return at which Gymnasium considers the task solved.
print(f'reward_threshold   : {env.spec.reward_threshold}')
print()

# --- Observation space: a Box in R^4 ---
print('=== Observation space ===')
# Python class of the space (Box = continuous, Discrete = integer, etc.).
print(f'type               : {type(env.observation_space).__name__}')
# Shape of one observation; here (4,) = four floats per state.
print(f'shape              : {env.observation_space.shape}')
# Numpy dtype of observations; CartPole uses float32.
print(f'dtype              : {env.observation_space.dtype}')
# Per-dimension lower bound of the observation Box.
print(f'low                : {env.observation_space.low}')
# Per-dimension upper bound of the observation Box.
print(f'high               : {env.observation_space.high}')
# Physical meaning of each of the 4 numbers in a state vector.
print('components         : [cart_position, cart_velocity, pole_angle_rad, pole_angular_velocity]')
print()

# --- Action space: a Discrete(2) ---
print('=== Action space ===')
# Class of the action space (Discrete here — integers 0..n-1).
print(f'type               : {type(env.action_space).__name__}')
# Number of possible actions.
print(f'n                  : {env.action_space.n}')
# What each action id means physically.
print('meaning            : 0 = push cart LEFT, 1 = push cart RIGHT')
print()

# --- Reward structure ---
print('=== Reward ===')
# Reward given every step the episode hasn't ended.
print(f'per step           : +1.0 while the pole stays upright')
# Declared reward range from the env; generic default, not the true range.
print(f'range              : {env.reward_range}')
print()

# --- One-step sample rollout to show the reset/step API ---
print('=== Sample rollout (1 random step) ===')
# Reset the env to a new starting state; seed makes it reproducible.
state, info = env.reset(seed=SEED)
# The starting observation the agent sees at t=0.
print(f'initial state      : {state}')
# Extra metadata dict from reset (usually empty for classic control).
print(f'reset info         : {info}')
# Sample a random action uniformly from the action space (just for demo).
action = env.action_space.sample()
# Advance the simulator one timestep with that action.
next_state, reward, terminated, truncated, info = env.step(action)
# Print the five return values of gymnasium's step() API.
print(f'action taken       : {action}')
print(f'next state         : {next_state}')
print(f'reward             : {reward}')
# terminated=True means the episode ended naturally (pole fell, cart left bounds).
print(f'terminated         : {terminated}  (pole fell / cart out of bounds)')
# truncated=True means we hit max_episode_steps without a natural end.
print(f'truncated          : {truncated}   (hit max_episode_steps)')
# Extra per-step metadata dict; CartPole leaves this empty.
print(f'step info          : {info}')

## 4. Policy network

A small MLP that maps state to action logits. `Categorical` gives us `sample()` and `log_prob()` in one object.

In [ ]:
class PolicyNet(nn.Module):
    # MLP that maps a state to action logits; sampling gives an action.
    def __init__(self, obs_dim, n_actions, hidden=128):
        super().__init__()                         # Initialize the nn.Module base.
        self.net = nn.Sequential(                  # Stack layers in order.
            nn.Linear(obs_dim, hidden),            # Linear: state vector -> hidden features.
            nn.Tanh(),                             # Nonlinearity (bounded, works well for small MLPs).
            nn.Linear(hidden, n_actions),          # Linear: hidden features -> one logit per action.
        )

    def forward(self, x):
        # Forward pass: states in -> action logits out (no softmax; Categorical handles it).
        return self.net(x)

    def act(self, state):
        # Convert a numpy state to a float32 tensor on the right device and add a batch dim.
        state = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        logits = self.forward(state)               # Run the network to get action logits.
        dist = Categorical(logits=logits)          # Build a categorical distribution over actions.
        action = dist.sample()                     # Sample one action (exploration via stochasticity).
        # Return the int action and the log-prob tensor (kept in the graph for the gradient update).
        return action.item(), dist.log_prob(action).squeeze(0)

## 5. Returns

Compute discounted returns $G_t$ backwards through the episode.

In [ ]:
def discounted_returns(rewards, gamma):
    # Compute G_t = r_t + gamma*r_{t+1} + gamma^2*r_{t+2} + ... for each t in the episode.
    returns = []                                 # Will hold G_0, G_1, ..., G_T.
    G = 0.0                                      # Running return, initialized at the terminal (t=T+1) as 0.
    for r in reversed(rewards):                  # Walk backwards so we can accumulate efficiently.
        G = r + gamma * G                        # Recurrence: G_t = r_t + gamma * G_{t+1}.
        returns.append(G)                        # Append in reverse time order (T, T-1, ..., 0).
    returns.reverse()                            # Flip back to forward time order (0, 1, ..., T).
    # Send returns to the training device as a float32 tensor for the policy loss.
    return torch.tensor(returns, dtype=torch.float32, device=device)

## 6. REINFORCE training loop

For each episode: roll out the policy, compute returns, and minimize

$$L(\theta) = -\frac{1}{T}\sum_t \log \pi_\theta(a_t\mid s_t)\; \hat{A}_t,$$

where $\hat{A}_t$ is the normalized return (mean 0, std 1) as a simple variance-reduction trick.

In [ ]:
def train_reinforce(env, episodes=600, gamma=0.99, lr=1e-2, log_every=25):
    # Vanilla REINFORCE: one gradient update per episode using Monte-Carlo returns.
    policy = PolicyNet(obs_dim, n_actions).to(device)     # Build a fresh policy on the chosen device.
    optimizer = torch.optim.Adam(policy.parameters(), lr=lr)  # Adam optimizer over policy params.

    history = []                                          # Per-episode total return, for plotting.
    recent = deque(maxlen=50)                             # Rolling window of the last 50 returns.

    for ep in range(1, episodes + 1):                     # Loop over episodes (1-indexed for logs).
        state, _ = env.reset(seed=SEED + ep)              # Reset env with a seed that varies per episode.
        log_probs, rewards = [], []                       # Collect log π(a_t|s_t) and r_t across the episode.
        done = False                                      # Episode-over flag (terminated OR truncated).
        while not done:                                   # Roll out until the episode ends.
            action, logp = policy.act(state)              # Sample an action and keep its log-prob.
            state, reward, terminated, truncated, _ = env.step(action)  # Advance one step.
            log_probs.append(logp)                        # Save log-prob for the gradient.
            rewards.append(reward)                        # Save reward for computing returns.
            done = terminated or truncated                # End on a natural end or step-limit.

        returns = discounted_returns(rewards, gamma)      # Compute G_t for every step of this episode.
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)  # Normalize returns (variance reduction).
        log_probs = torch.stack(log_probs)                # List of tensors -> single 1D tensor.
        loss = -(log_probs * returns).mean()              # Policy gradient loss: -E[log π(a|s) * G_t].

        optimizer.zero_grad()                             # Clear gradients from the previous update.
        loss.backward()                                   # Backprop through all cached log-probs.
        optimizer.step()                                  # Apply the Adam update to policy params.

        total = sum(rewards)                              # Episode return (sum of per-step rewards).
        history.append(total)                             # Record for the learning curve.
        recent.append(total)                              # Track for the moving average.
        if ep % log_every == 0:                           # Log progress every `log_every` episodes.
            print(f'ep {ep:4d} | return {total:6.1f} | avg50 {np.mean(recent):6.1f} | loss {loss.item():+.3f}')

    return policy, history                                # Return the trained policy and return curve.

In [ ]:
# Kick off training and capture the trained policy plus its return-per-episode history.
policy, history = train_reinforce(env)

## 7. Learning curve

In [ ]:
def plot_history(history, title='REINFORCE on CartPole-v1'):
    window = 25                                                           # Smoothing window (episodes).
    smoothed = np.convolve(history, np.ones(window) / window, mode='valid')  # Simple moving average.
    plt.figure(figsize=(9, 4))                                            # Create a new 9x4 inch figure.
    plt.plot(history, alpha=0.3, label='episode return')                  # Raw per-episode returns (light).
    plt.plot(range(window - 1, len(history)), smoothed,                   # Smoothed curve aligned to its window.
             label=f'{window}-ep moving avg')
    plt.xlabel('episode')                                                 # X-axis: episode index.
    plt.ylabel('return')                                                  # Y-axis: total reward per episode.
    plt.title(title)                                                      # Figure title.
    plt.legend()                                                          # Show which line is which.
    plt.grid(alpha=0.3)                                                   # Faint grid for readability.
    plt.show()                                                            # Render the figure inline.

plot_history(history)                                                     # Plot the vanilla REINFORCE curve.

## 8. REINFORCE with a learned value baseline

Using $\hat{A}_t = G_t - V_\phi(s_t)$ reduces variance without adding bias. We fit $V_\phi$ by regression against $G_t$.

In [ ]:
class ValueNet(nn.Module):
    # Small MLP that estimates V(s): the expected return from state s under the current policy.
    def __init__(self, obs_dim, hidden=128):
        super().__init__()                                # Initialize nn.Module.
        self.net = nn.Sequential(                         # Layer stack.
            nn.Linear(obs_dim, hidden),                   # state -> hidden features.
            nn.Tanh(),                                    # Nonlinearity.
            nn.Linear(hidden, 1),                         # hidden -> scalar value.
        )

    def forward(self, x):
        # Returns shape (batch,) — the trailing size-1 dim is squeezed off.
        return self.net(x).squeeze(-1)


def train_reinforce_with_baseline(env, episodes=600, gamma=0.99, lr_pi=1e-2, lr_v=1e-2, log_every=25):
    # REINFORCE with a learned value baseline: advantage = G_t - V(s_t), for lower gradient variance.
    policy = PolicyNet(obs_dim, n_actions).to(device)     # Fresh policy.
    value = ValueNet(obs_dim).to(device)                  # Fresh value function.
    opt_pi = torch.optim.Adam(policy.parameters(), lr=lr_pi)  # Optimizer for the policy.
    opt_v = torch.optim.Adam(value.parameters(), lr=lr_v)     # Separate optimizer for the value net.

    history = []                                          # Per-episode returns for plotting.
    recent = deque(maxlen=50)                             # Rolling 50-episode window.

    for ep in range(1, episodes + 1):                     # Loop over training episodes.
        state, _ = env.reset(seed=SEED + ep)              # Reproducible per-episode reset.
        states, log_probs, rewards = [], [], []           # Also collect states now (needed for V(s_t)).
        done = False
        while not done:
            action, logp = policy.act(state)              # Sample action + grab its log-prob.
            states.append(state)                          # Record the state BEFORE stepping.
            log_probs.append(logp)                        # Record the log-prob for the policy loss.
            state, reward, terminated, truncated, _ = env.step(action)  # Advance one step.
            rewards.append(reward)                        # Record the reward for the return.
            done = terminated or truncated                # Standard end condition.

        returns = discounted_returns(rewards, gamma)      # G_t for every step in the episode.
        # Stack states into a (T, obs_dim) float32 tensor on the right device for batched V(s).
        states_t = torch.as_tensor(np.array(states), dtype=torch.float32, device=device)
        values = value(states_t)                          # V(s_t) for every t; shape (T,).

        advantages = (returns - values).detach()          # A_t = G_t - V(s_t); detach so no value-net grad flows here.
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)  # Normalize advantages.

        log_probs = torch.stack(log_probs)                # (T,) tensor of log-probs.
        pi_loss = -(log_probs * advantages).mean()        # Policy loss: -E[log π(a|s) * A_t].
        v_loss = F.mse_loss(values, returns)              # Value loss: regress V(s_t) toward G_t.

        opt_pi.zero_grad(); pi_loss.backward(); opt_pi.step()  # Update the policy.
        opt_v.zero_grad(); v_loss.backward(); opt_v.step()     # Update the value net.

        total = sum(rewards)                              # Episode return.
        history.append(total)                             # For the learning curve.
        recent.append(total)                              # For the moving average.
        if ep % log_every == 0:                           # Log every N episodes.
            print(f'ep {ep:4d} | return {total:6.1f} | avg50 {np.mean(recent):6.1f} | pi {pi_loss.item():+.3f} | v {v_loss.item():.2f}')

    return policy, value, history                         # Return both networks plus the return history.

In [ ]:
# Train the baseline variant and capture the trained policy, value net, and return history.
policy_b, value_b, history_b = train_reinforce_with_baseline(env)
# Plot the learning curve for the baseline variant.
plot_history(history_b, title='REINFORCE + value baseline on CartPole-v1')

## 9. Evaluate the trained policy

Run greedy (argmax) and stochastic rollouts and report mean return.

In [ ]:
def evaluate(policy, env, episodes=20, greedy=True):
    # Roll out the trained policy for `episodes` episodes and return mean/std of total reward.
    returns = []                                                 # Collect total reward per eval episode.
    for i in range(episodes):                                    # Loop over eval episodes.
        state, _ = env.reset(seed=10_000 + i)                    # Distinct seeds from training (10_000+...).
        done = False
        total = 0.0                                              # Running episode return.
        while not done:
            # Convert state to float32 tensor on device with batch dim (1, obs_dim).
            state_t = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
            with torch.no_grad():                                # No grads during evaluation — saves memory.
                logits = policy(state_t)                         # Get action logits.
            if greedy:
                action = int(torch.argmax(logits, dim=-1).item())  # Deterministic: pick the highest-logit action.
            else:
                action = int(Categorical(logits=logits).sample().item())  # Stochastic: sample as in training.
            state, reward, terminated, truncated, _ = env.step(action)    # Step the env.
            total += reward                                      # Accumulate reward.
            done = terminated or truncated                       # Standard end condition.
        returns.append(total)                                    # Record this episode's return.
    return float(np.mean(returns)), float(np.std(returns))       # Mean and std across eval episodes.

# Greedy eval: pure exploitation of what the policy has learned.
mean_g, std_g = evaluate(policy_b, env, greedy=True)
# Stochastic eval: samples actions — closer to training-time behavior.
mean_s, std_s = evaluate(policy_b, env, greedy=False)
print(f'greedy    : {mean_g:.1f} ± {std_g:.1f}')                 # Report mean ± std for greedy.
print(f'stochastic: {mean_s:.1f} ± {std_s:.1f}')                 # Report mean ± std for stochastic.

## 10. Where to go next

- **Actor-Critic / A2C** — bootstrap $V_\phi$ to form a TD advantage instead of using full Monte-Carlo returns.
- **GAE** (Generalized Advantage Estimation) — interpolates between TD and Monte-Carlo.
- **PPO** — clipped surrogate objective for stable large updates; today's default policy-gradient algorithm.
- **Continuous actions** — replace `Categorical` with `Normal` and output mean/log-std.
- **Harder envs** — try `LunarLander-v2`, `Acrobot-v1`, or MuJoCo tasks.